# Milestone 4 lab, notebook 4 of 7: A loop or a set

## Where we are

| Notebook | What you do |
|---|---|
| 1. Load first, transform later | Python stops transforming. Compare ELT with milestone 1's ETL |
| 2. Your first model | Write one SELECT in a file, and let dbt build it |
| 3. Clean, join, enrich | One row per order again, then two joins and their traps |
| **4. A loop or a set** (you are here) | Answer one question row by row, then with one statement |
| 5. Layers and the incremental model | Three layers, and a report that rebuilds only its newest days |
| 6. Tests on the tables | Break the data on purpose, and watch the pipeline refuse it |
| 7. The production version | One morning as ELT: Python loads, dbt transforms and tests |

There are two ways to think about a calculation over many rows. You can visit each row and do something. Or you can describe the result you want from all the rows together. This short notebook does one job both ways, and measures them.

In [1]:
import time

from lab_helpers import dbt, lab_warehouse, load_only, money, table

con = lab_warehouse(fresh=True)
load_only(con, "2026-06-08", "2026-06-09", "2026-06-10", "2026-06-11", "2026-06-12")
dbt("build", show=False)
orders_in_table = con.execute("SELECT count(*) FROM core.orders").fetchone()[0]
print(f"{orders_in_table} orders are in core.orders.")

Landed 226 raw rows for the morning of 2026-06-08.
Landed 102 raw rows for the morning of 2026-06-09.
Landed  93 raw rows for the morning of 2026-06-10.
Landed 120 raw rows for the morning of 2026-06-11.
Landed 103 raw rows for the morning of 2026-06-12.


347 orders are in core.orders.


## The job

Work out the revenue of each business day: the report, without the cities.

## Way 1: row by row

This is how a program is usually written. Get the list of orders. For each one, go back to the database for its details, and add to a running total.

In [2]:
def report_row_by_row():
    totals, trips = {}, 0
    order_ids = [row[0] for row in con.execute("SELECT order_id FROM core.orders")]
    trips += 1
    for order_id in order_ids:
        day, status, net_cents, deleted = con.execute(
            "SELECT order_date, status, net_cents, is_deleted FROM core.orders WHERE order_id = %s", [order_id]
        ).fetchone()
        trips += 1
        if status != "cancelled" and not deleted:
            totals[day] = totals.get(day, 0) + net_cents
    return totals, trips

started = time.perf_counter()
by_row, trips_by_row = report_row_by_row()
seconds_by_row = time.perf_counter() - started

print(f"Row by row: {trips_by_row} trips to the database, {seconds_by_row * 1000:.0f} milliseconds.")

Row by row: 348 trips to the database, 59 milliseconds.


## Way 2: set-based

Say **what** you want from all the rows together, and let the warehouse work out how.

In [3]:
def report_set_based():
    rows = con.execute("""
        SELECT order_date, sum(net_cents)
        FROM core.orders
        WHERE status <> 'cancelled' AND NOT is_deleted
        GROUP BY order_date
    """).fetchall()
    return {day: int(cents) for day, cents in rows}, 1

started = time.perf_counter()
by_set, trips_by_set = report_set_based()
seconds_by_set = time.perf_counter() - started

print(f"Set-based:  {trips_by_set} trip to the database, {seconds_by_set * 1000:.0f} milliseconds.")

Set-based:  1 trip to the database, 0 milliseconds.


In [4]:
print("Do both give the same answer?", by_row == by_set)
print(f"Trips to the database: {trips_by_row} against {trips_by_set}.")
print(f"On this run the loop took about {seconds_by_row / seconds_by_set:.0f} times as long.")

Do both give the same answer? True
Trips to the database: 348 against 1.
On this run the loop took about 130 times as long.


The same answer. The times are from this run, on this computer, with a database on the same machine. Yours will differ a little each time you run the cell.

What does not differ is the **shape**. The loop makes one trip per order. With 347 orders that is 348 trips. With 3 million orders it is 3 million trips, and each trip has a cost that no faster computer removes. The set-based query makes one trip, however many orders there are.

## Why it matters beyond speed

| | Row by row | Set-based |
|---|---|---|
| You write | **How**: a loop, a variable, an `if` | **What**: the result you want |
| Who does the work | Your program, one row at a time | The warehouse, which is built for it |
| Trips to the database | One per row | One |
| Where a rule hides | Inside an `if`, inside a loop | In the `WHERE` and `GROUP BY`, in plain sight |

Every dbt model is set-based: it is one `SELECT`. That is one more reason the rules moved out of Python.

The worst case is the one you just wrote: **a loop that calls the database once per row.** If you see it in a pipeline, replace it with a join or a `GROUP BY`.

In [5]:
con.close()

## New words

| Word | Meaning | Here |
|---|---|---|
| **Row-by-row** | "Row-by-row code visits one row at a time." | The loop: one query per order |
| **Set-based** | "Set-based code says what you want from all the rows together." | One `GROUP BY` |

The definitions are from the D8Loop chapter [Transformation Fundamentals](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/transformation-fundamentals).

## What you now know

- The loop and the `GROUP BY` gave the same answer.
- The loop made one trip to the database per order. The set-based query made one trip.
- Set-based SQL says what you want, and shows the rule in plain sight.

**Say it in an interview:** "I push work to the engine. I measured it on my own project: a Python loop with one query per order made 348 trips to the database, and one GROUP BY made one. The answer was identical. The number of trips grows with the data, and that is what decides whether a job finishes."

Next: [notebook 5](05_layers_and_the_incremental_model.ipynb) looks at how the models are arranged, and at the model that does not rebuild everything.